# City Library Management System

In [ ]:
from datetime import datetime



# Book Model (Now with Copies)


class Book:
    def __init__(self, book_id, title, author, genre, total_copies):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.genre = genre
        self.total_copies = total_copies
        self.available_copies = total_copies

    def __repr__(self):
        return (f"{self.book_id} | {self.title} | {self.author} | {self.genre} | "
                f"Available: {self.available_copies}/{self.total_copies}")


# Member Model (Tracks quantity per book)

class Member:
    MAX_TOTAL_BORROW = 5

    def __init__(self, member_id, name, age, contact):
        self.member_id = member_id
        self.name = name
        self.age = age
        self.contact = contact
        self.borrowed_books = {}  # {book_id: quantity}

    def total_books_borrowed(self):
        return sum(self.borrowed_books.values())

    def can_borrow(self, quantity):
        return self.total_books_borrowed() + quantity <= self.MAX_TOTAL_BORROW

    def __repr__(self):
        return f"{self.member_id} | {self.name} | Borrowed: {self.borrowed_books}"



# Transaction Log


class Transaction:
    def __init__(self, member_id, book_id, quantity, action):
        self.member_id = member_id
        self.book_id = book_id
        self.quantity = quantity
        self.action = action
        self.timestamp = datetime.now()

    def __repr__(self):
        return (f"{self.timestamp.strftime('%Y-%m-%d %H:%M:%S')} | "
                f"Member {self.member_id} | Book {self.book_id} | "
                f"{self.action} | Qty: {self.quantity}")



# Library System

class Library:
    def __init__(self):
        self.books = {}
        self.members = {}
        self.borrow_log = []
        self.genre_count = {}

    
    # Add Book / Member
    

    def add_book(self, book_id, title, author, genre, total_copies):
        if book_id in self.books:
            print("Error: Duplicate Book ID.")
            return
        if total_copies <= 0:
            print("Error: Total copies must be > 0.")
            return

        self.books[book_id] = Book(book_id, title, author, genre, total_copies)
        print("Book added successfully.")

    def add_member(self, member_id, name, age, contact):
        if member_id in self.members:
            print("Error: Duplicate Member ID.")
            return
        self.members[member_id] = Member(member_id, name, age, contact)
        print("Member added successfully.")

    # Borrow System (With Quantity)
   

    def issue_book(self, member_id, book_id, quantity):

        if book_id not in self.books:
            print("Error: Book not found.")
            return

        if member_id not in self.members:
            print("Error: Member not found.")
            return

        book = self.books[book_id]
        member = self.members[member_id]

        if quantity <= 0:
            print("Error: Quantity must be positive.")
            return

        if book.available_copies < quantity:
            print("Error: Not enough copies available.")
            return

        if not member.can_borrow(quantity):
            print("Error: Borrow limit exceeded.")
            return

        # Update states
        book.available_copies -= quantity
        member.borrowed_books[book_id] = member.borrowed_books.get(book_id, 0) + quantity

        self.borrow_log.append(Transaction(member_id, book_id, quantity, "Borrow"))
        self.genre_count[book.genre] = self.genre_count.get(book.genre, 0) + quantity

        print("Book(s) issued successfully.")

    def return_book(self, member_id, book_id, quantity):

        if book_id not in self.books or member_id not in self.members:
            print("Error: Invalid book or member.")
            return

        member = self.members[member_id]
        book = self.books[book_id]

        if book_id not in member.borrowed_books:
            print("Error: Member did not borrow this book.")
            return

        if quantity <= 0 or quantity > member.borrowed_books[book_id]:
            print("Error: Invalid return quantity.")
            return

        # Update states
        book.available_copies += quantity
        member.borrowed_books[book_id] -= quantity

        if member.borrowed_books[book_id] == 0:
            del member.borrowed_books[book_id]

        self.borrow_log.append(Transaction(member_id, book_id, quantity, "Return"))

        print("Book(s) returned successfully.")

   
    # Reports
   

    def search_by_title(self, keyword):
        results = [book for book in self.books.values()
                   if keyword.lower() in book.title.lower()]
        self._print_results(results)

    def search_by_author(self, keyword):
        results = [book for book in self.books.values()
                   if keyword.lower() in book.author.lower()]
        self._print_results(results)

    def available_books_by_genre(self, genre):
        results = [book for book in self.books.values()
                   if book.genre.lower() == genre.lower() and book.available_copies > 0]
        self._print_results(results)

    def members_with_active_loans(self):
        results = [member for member in self.members.values()
                   if member.borrowed_books]
        self._print_results(results)

    def most_popular_genre(self):
        if not self.genre_count:
            print("No books issued yet.")
            return
        popular = max(self.genre_count, key=self.genre_count.get)
        print("Most Popular Genre:", popular)

    def library_summary(self):
        print("Total Books:", len(self.books))
        print("Total Members:", len(self.members))
        total_active = sum(m.total_books_borrowed() for m in self.members.values())
        print("Total Active Borrowed Copies:", total_active)
        self.most_popular_genre()

    def show_borrow_log(self):
        for log in self.borrow_log:
            print(log)

    def _print_results(self, results):
        if not results:
            print("No results found.")
            return
        for item in results:
            print(item)


#Interface


def main():
    library = Library()

    while True:
        print("\n===== CITY LIBRARY SYSTEM =====")
        print("1. Add Book")
        print("2. Add Member")
        print("3. Issue Book")
        print("4. Return Book")
        print("5. Search by Title")
        print("6. Search by Author")
        print("7. Available Books by Genre")
        print("8. Members with Active Loans")
        print("9. Library Summary")
        print("10. Show Borrow Log")
        print("0. Exit")

        choice = input("Enter choice: ")

        if choice == "1":
            library.add_book(
                int(input("Book ID: ")),
                input("Title: "),
                input("Author: "),
                input("Genre: "),
                int(input("Total Copies: "))
            )

        elif choice == "2":
            library.add_member(
                int(input("Member ID: ")),
                input("Name: "),
                int(input("Age: ")),
                input("Contact: ")
            )

        elif choice == "3":
            library.issue_book(
                int(input("Member ID: ")),
                int(input("Book ID: ")),
                int(input("Quantity: "))
            )

        elif choice == "4":
            library.return_book(
                int(input("Member ID: ")),
                int(input("Book ID: ")),
                int(input("Quantity: "))
            )

        elif choice == "5":
            library.search_by_title(input("Enter title keyword: "))

        elif choice == "6":
            library.search_by_author(input("Enter author keyword: "))

        elif choice == "7":
            library.available_books_by_genre(input("Enter genre: "))

        elif choice == "8":
            library.members_with_active_loans()

        elif choice == "9":
            library.library_summary()

        elif choice == "10":
            library.show_borrow_log()

        elif choice == "0":
            print("Exiting system.")
            break

        else:
            print("Invalid choice.")


if __name__ == "__main__":
    main()